# CTA策略工厂类 WtStraFact
```cpp
class WtStraFact : public ICtaStrategyFact
```
实现了 ICtaStrategyFact 接口，是 CTA 策略的工厂类。负责创建、管理和删除CTA策略实例，支持策略的动态加载和插件化开发。

## 获取工厂名称 getName
```cpp
/**
 * @brief 获取工厂名称的实现
 * @return const char* 返回策略工厂的名称字符串
 * 该函数返回策略工厂的名称，用于标识和管理不同的策略工厂。
 * @note 返回的是常量字符串指针，不需要调用者释放内存
 */
const char* WtStraFact::getName()
{
	return FACT_NAME; // 返回工厂名称常量
}
```

## 创建策略实例 createStrategy
```cpp
/**
 * @brief 创建策略实例的实现
 * @param name 策略名称，用于指定要创建的策略类型
 * @param id 策略唯一标识符，用于在系统中唯一标识该策略实例
 * @return CtaStrategy* 返回创建的策略对象指针，如果策略名称不存在则返回NULL
 * 
 * 该函数根据策略名称创建对应的策略对象实例。
 * 当前支持的策略类型：
 * - "DualThrust": 创建DualThrust双突破策略实例
 * 
 * 如果传入的策略名称不匹配任何已知策略，则返回NULL。
 * @note 调用者负责管理返回的指针，使用完毕后应通过deleteStrategy删除
 */
CtaStrategy* WtStraFact::createStrategy(const char* name, const char* id)  // 创建策略函数实现
{
	if (strcmp(name, "DualThrust") == 0)
		return new WtStraDualThrust(id);
	return NULL;
}
```

## 枚举策略名称 enumStrategy
```cpp
/**
 * @brief 枚举策略名称的实现
 * @param cb 枚举策略名称的回调函数，每枚举到一个策略都会调用此回调
 * 
 * 该函数枚举工厂中所有可用的策略类型，通过回调函数通知调用者。
 * 当前工厂支持的策略：
 * - "DualThrust": DualThrust双突破策略
 * 
 * 回调函数会被调用一次，传入以下参数：
 * - factName: 工厂名称（"WtCtaStraFact"）
 * - straName: 策略名称（"DualThrust"）
 * - isLast: 是否为最后一个策略（true，因为只有一个策略）
 * 
 * @note 如果将来添加更多策略，需要在此函数中添加更多的回调调用
 */
void WtStraFact::enumStrategy(FuncEnumStrategyCallback cb)
{
	cb(FACT_NAME, "DualThrust", true); // 调用回调函数，传入工厂名称、策略名称和是否为最后一个策略
}
```

## 删除策略实例 deleteStrategy
```cpp
/**
 * @brief 删除策略实例的实现
 * @param stra 要删除的策略对象指针
 * @return bool 删除成功返回true，失败返回false
 * 
 * 该函数删除指定的策略对象，释放相关资源。
 * 删除前会进行以下检查：
 * 1. 检查策略指针是否为空，如果为空则直接返回true（视为成功）
 * 2. 检查策略是否属于本工厂创建，通过比较策略的工厂名称
 * 3. 只有属于本工厂的策略才会被删除，其他策略返回false
 * 
 * @note 删除后策略指针将失效，调用者不应再使用该指针
 */
bool WtStraFact::deleteStrategy(CtaStrategy* stra)  // 删除策略函数实现
{
	if (stra == NULL)
		return true;

	if (strcmp(stra->getFactName(), FACT_NAME) != 0)  // 检查策略是否属于本工厂创建
		return false;  // 如果不属于本工厂，返回false（拒绝删除）

	delete stra;  // 删除策略对象，调用析构函数释放资源
	return true;
}
```

# 双突破策略类 WtStraDualThrust
```cpp
class WtStraDualThrust : public CtaStrategy
```

## 成员
- **策略指标参数**
  - `double _k1`：上轨系数，用于计算上突破轨道，通常取值范围为0.5-1.5
  - `double _k2`：下轨系数，用于计算下突破轨道，通常取值范围为0.5-1.5
  - `uint32_t _days`：回看天数，用于计算价格波动范围，通常取值为5-20
  - `std::string _moncode`：当前主力合约代码，用于期货合约的自动换月处理
- **数据周期相关参数**
  - `std::string _period`：K线周期，如"m1"（1分钟）、"m5"（5分钟）、"d1"（日线）等
  - `uint32_t _count`：K线条数，用于获取历史K线数据，通常取值为_days的2-3倍
- **合约相关参数**
  - `std::string _code`：合约代码，策略交易的合约代码（可能是连续合约代码）
  - `bool _isstk`：是否为股票标志，true表示股票模式（不支持做空），false表示期货模式（支持做空）

## 获取所属策略工厂名称 getFactName
```cpp
/**
 * @brief 获取所属策略工厂名称的实现
 * @return const char* 返回策略所属的工厂名称字符串
 * 
 * 该函数返回策略所属的策略工厂名称，用于标识策略的来源工厂。
 * 返回值为"WtCtaStraFact"，表示该策略由WtCtaStraFact工厂创建。
 * 
 * @note FACT_NAME常量定义在WtCtaStraFact.cpp中，值为"WtCtaStraFact"
 */
const char* WtStraDualThrust::getFactName()
{
	return FACT_NAME;
}
```

## 获取策略名称 getName
```cpp
/**
 * @brief 获取策略名称的实现
 * @return const char* 返回策略的名称字符串
 * 
 * 该函数返回策略的名称，用于标识策略类型。
 * 返回值为"DualThrust"，表示这是DualThrust双突破策略。
 */
const char* WtStraDualThrust::getName()
{
	return "DualThrust";
}
```

## 策略初始化 init
```cpp
/**
 * @brief 策略初始化实现
 * @param cfg 策略配置参数，包含策略运行所需的所有参数
 * @return bool 初始化成功返回true，失败返回false
 * 
 * 该函数从配置参数中加载策略运行所需的参数，包括：
 * - days: 回看天数，用于计算价格波动范围（必需参数）
 * - k1: 上轨系数，用于计算上突破轨道（必需参数）
 * - k2: 下轨系数，用于计算下突破轨道（必需参数）
 * - period: K线周期，如"m1"、"m5"、"d1"等（必需参数）
 * - count: K线条数，用于获取历史K线数据（必需参数）
 * - code: 合约代码，策略交易的合约（必需参数）
 * - stock: 是否为股票，true表示股票模式（不支持做空），false表示期货模式（可选参数，默认为false）
 * 
 * @note 如果配置参数为空，函数返回false
 * @note 如果配置参数中缺少必要参数，可能导致运行时错误
 */
bool WtStraDualThrust::init(WTSVariant* cfg)  // 策略初始化函数实现
{
	if (cfg == NULL)
		return false;

	_days = cfg->getUInt32("days");
	_k1 = cfg->getDouble("k1");
	_k2 = cfg->getDouble("k2");

	_period = cfg->getCString("period");
	_count = cfg->getUInt32("count");
	_code = cfg->getCString("code");

	_isstk = cfg->getBoolean("stock");

	return true;
}
```

## 策略调度执行入口 on_schedule
策略的核心驱动逻辑。这是 CTA 策略的**主循环**，通常在 K 线闭合时被调度调用。它负责获取数据、计算指标、生成信号并执行交易指令。
* **数据获取与校验**
  * 调用 `ctx->stra_get_bars` 拉取指定周期（如 "m5"）和长度（`_count`）的历史 K 线数据。
  * **安全检查**：如果 K 线为空或长度为 0，直接释放内存并返回，防止空指针崩溃。
* **指标计算 (DualThrust 核心算法)**
  * **确定回溯窗口**：使用 `_days` 参数确定计算 range 的时间跨度。
  * **计算极值**：
    * `hh` (High High)：过去 N 天的最高价。
    * `ll` (Low Low)：过去 N 天的最低价。
    * `hc` (High Close)：过去 N 天收盘价的最高值。
    * `lc` (Low Close)：过去 N 天收盘价的最低值。
  * **计算波动区间 (Range)**：`Range = max(hh - lc, hc - ll)`。
  * **计算轨道**：
    * 上轨 (`upper_bound`) = 当前 K 线开盘价 + `_k1` * Range。
    * 下轨 (`lower_bound`) = 当前 K 线开盘价 - `_k2` * Range。
  * **UI 更新**：调用 `ctx->set_index_value` 将计算出的上下轨数值绑定到图表指标上。
* **交易信号生成与执行**
  * 获取当前持仓 `curPos` 和当前价格 `curPx`。
  * **空仓状态 (Pos == 0)**：
    * 如果 `价格 >= 上轨`：开多仓 (`stra_enter_long`)，添加图表买入标记。
    * 如果 `价格 <= 下轨` 且**非股票模式**：开空仓 (`stra_enter_short`)，添加图表卖出标记。
  * **持多仓状态 (Pos > 0)**：
    * 如果 `价格 <= 下轨`：平多仓 (`stra_exit_long`)，即止损或反转出场。
  * **持空仓状态 (Pos < 0)**：
    * 如果 `价格 >= 上轨` 且**非股票模式**：平空仓 (`stra_exit_short`)。
* **资源清理**
  * **关键步骤**：最后必须调用 `kline->release()` 释放 K 线数据对象的内存，否则会导致严重的内存泄漏。
```cpp
/**
 * @brief 策略调度执行入口实现
 * @param ctx 策略上下文对象，提供数据访问和交易执行接口
 * @param curDate 当前日期，格式为YYYYMMDD
 * @param curTime 当前时间，格式为HHMMSS
 */
void WtStraDualThrust::on_schedule(ICtaStraCtx* ctx, uint32_t curDate, uint32_t curTime) 
```

## 策略初始化完成回调 on_init
策略生命周期的**初始化阶段**。在策略加载参数之后、正式运行之前调用。用于建立数据订阅、预加载数据和配置图表显示。
* **订阅实时数据**
  * 调用 `ctx->stra_sub_ticks` 订阅合约的 Tick 数据，确保策略运行期间能收到行情驱动。
* **数据预热与校验**
  * 尝试调用 `ctx->stra_get_bars` 读取历史 K 线。
  * 这一步主要用于检查数据流是否通畅以及回测/实盘环境是否准备好历史数据。
  * 读取后立即调用 `kline->release()` 释放，因为此处只做检查，不涉及逻辑计算。
* **图表与指标注册 (UI 配置)**
  * **设置 K 线图**：`ctx->set_chart_kline` 告诉前端/GUI 该策略关注的 K 线周期。
  * **注册指标组**：`ctx->register_index` 创建名为 "DualThrust" 的指标组。
  * **注册指标线**：`ctx->register_index_line` 在该组下定义两条线 "upper_bound"（上轨）和 "lower_bound"（下轨），以便在 `on_schedule` 中写入数值并在界面绘制。
```cpp
/**
 * @brief 策略初始化完成回调实现
 * @param ctx 策略上下文对象，提供数据访问和交易执行接口
 * @note 如果K线数据获取失败，函数会提前返回，但不会影响策略运行
 */
void WtStraDualThrust::on_init(ICtaStraCtx* ctx)
```

## Tick数据处理回调 on_tick
高频数据驱动入口：当订阅的合约有新的 Tick（快照）数据到达时触发。
```cpp
/**
 * @brief Tick数据处理回调实现
 * @param ctx 策略上下文对象，提供数据访问和交易执行接口
 * @param stdCode 标准合约代码，触发Tick数据的合约
 * @param newTick 新的Tick数据，包含最新的价格和成交量信息
 * 
 * 该函数在接收到新的Tick数据时被调用，用于处理实时市场数据。
 * DualThrust策略主要基于K线数据进行交易决策，因此Tick数据处理为空实现。
 * 
 * 如果将来需要基于Tick数据做更精细的交易决策（如：
 * - 基于Tick级别的价格变化进行更精确的入场/出场
 * - 基于Tick级别的成交量进行过滤
 * - 基于Tick级别的买卖盘口进行决策），可以在此函数中实现。
 * 
 * @note 该函数虽然为空实现，但必须保留，因为它是基类的虚函数
 */
void WtStraDualThrust::on_tick(ICtaStraCtx* ctx, const char* stdCode, WTSTickData* newTick) {}
```

## 交易日开始回调 on_session_begin
交易日/会话开始时的处理钩子：主要用于处理**期货主力合约换月**逻辑。
* **获取主力合约映射**
  * 调用 `ctx->stra_get_rawcode`。如果策略配置的是主力连续代码（如 `IF.hot`），该函数会返回当前日期对应的真实合约代码（如 `IF2106`）。
* **检测换月**
  * 比较新获取的 `newMonCode` 和策略内部记录的 `_moncode`。
  * 如果两者不一致，说明发生了主力合约切换。
* **执行移仓**
  * **检查旧仓位**：如果旧主力合约 (`_moncode`) 上有持仓 (`curPos != 0`)。
  * **记录日志**：输出日志提示即将进行换月操作。
  * **虚拟移仓**：
    * `ctx->stra_set_position(..., 0, "switchout")`：将旧合约持仓强制设为 0（标记为换出）。
    * `ctx->stra_set_position(..., curPos, "switchin")`：在新合约上设置同方向、同数量的持仓（标记为换入）。
    * *注意*：这里的 `set_position` 在回测中通常直接修改持仓记录，在实盘中可能触发实际的平仓和开仓指令（取决于底层执行单元的配置）。
* **更新状态**
  * 将 `_moncode` 更新为 `newMonCode`，以便后续交易日使用。
```cpp
/**
 * @brief 交易日开始回调实现
 * @param ctx 策略上下文对象，提供数据访问和交易执行接口
 * @param uTDate 交易日日期，格式为YYYYMMDD
 * @note 换月操作不会改变持仓方向，只是将持仓从旧主力转移到新主力
 */
void WtStraDualThrust::on_session_begin(ICtaStraCtx* ctx, uint32_t uTDate)
```